# Comparação entre as Arquiteturas

Neste notebook é realizada uma comparação entre as soluções desenvolvidas:

- Trilha A → Data Warehouse (Modelo Estrela + DuckDB)
- Trilha B → Data Lakehouse (Bronze, Silver, Gold + Delta Lake)

In [23]:
import duckdb
import pandas as pd
import time

trilha A

In [24]:
con = duckdb.connect("trilha A/dw_projeto.duckdb")

trilha B

In [25]:
gold_uf = pd.read_parquet(
    "trilha B/datalake/gold/gold_uf.parquet"
)

gold_mensal = pd.read_parquet(
    "trilha B/datalake/gold/gold_mensal.parquet"
)

gold_causa = pd.read_parquet(
    "trilha B/datalake/gold/gold_causa.parquet"
)

gold_municipio = pd.read_parquet(
    "trilha B/datalake/gold/gold_municipio.parquet"
)

gold_dia = pd.read_parquet(
    "trilha B/datalake/gold/gold_dia_semana.parquet"
)

gold_tipo = pd.read_parquet(
    "trilha B/datalake/gold/gold_tipo_acidente.parquet"
)

# Pergunta 1 

Quais estados possuem mais acidentes?

trilha A

In [26]:
inicio = time.time()

consulta = """

SELECT

    l.uf,

    COUNT(*) AS acidentes

FROM fato_acidentes f

JOIN dim_local l

ON f.sk_local=l.sk_local

GROUP BY l.uf

ORDER BY acidentes DESC

"""

dw_uf = con.execute(consulta).fetchdf()

tempo_dw = time.time()-inicio

dw_uf.head()

,uf,acidentes
0,MG,2985
1,SC,2847
2,PR,2561
3,RJ,2099
4,RS,1488


trilha B

In [27]:
inicio = time.time()

dl_uf = gold_uf.sort_values(
    "acidentes",
    ascending=False
)

tempo_dl = time.time()-inicio

dl_uf.head()

,uf,acidentes,mortos,feridos,veiculos
10,MG,2985,269,3816,5781
23,SC,2847,138,3341,5544
17,PR,2561,192,2932,5217
18,RJ,2099,122,2493,3909
22,RS,1488,108,1703,2853


# Comparação

In [28]:
print("Tempo DW:", tempo_dw)
print("Tempo DL:", tempo_dl)

Tempo DW: 0.006862640380859375
Tempo DL: 0.0048487186431884766


# Pergunta 2 

Como os acidentes evoluem ao longo do tempo?

trilha A

In [29]:
inicio = time.time()

consulta = """

SELECT

    d.ano,
    d.mes,
    COUNT(*) AS acidentes,
    SUM(f.mortos) AS mortos

FROM fato_acidentes f

JOIN dim_data d

ON f.sk_data = d.sk_data

GROUP BY d.ano, d.mes

ORDER BY d.ano, d.mes

"""

dw_mensal = con.execute(consulta).fetchdf()

tempo_dw = time.time() - inicio

dw_mensal.head()

,ano,mes,acidentes,mortos
0,2026,1,5822,494.0
1,2026,2,5591,445.0
2,2026,3,6119,469.0
3,2026,4,5943,510.0


trilha B

In [30]:
inicio = time.time()

dl_mensal = gold_mensal.copy()

tempo_dl = time.time() - inicio

dl_mensal.head()

,ano,mes,acidentes,mortos,feridos
0,2026,1,5822,494,7210
1,2026,2,5591,445,6785
2,2026,3,6119,469,6717
3,2026,4,5943,510,6845


# Comparação

In [31]:
print("Tempo DW:", tempo_dw)
print("Tempo DL:", tempo_dl)

Tempo DW: 0.0036771297454833984
Tempo DL: 0.0002760887145996094


# Pergunta 3

Quais são as principais causas?

trilha A

In [32]:
inicio = time.time()

consulta = """

SELECT

    a.causa_acidente,

    COUNT(*) AS acidentes,

    SUM(f.mortos) AS mortos

FROM fato_acidentes f

JOIN dim_acidente a

ON f.sk_acidente = a.sk_acidente

GROUP BY a.causa_acidente

ORDER BY acidentes DESC

"""

dw_causa = con.execute(consulta).fetchdf()

tempo_dw = time.time() - inicio

dw_causa.head()

,causa_acidente,acidentes,mortos
0,AUSÊNCIA DE REAÇÃO DO CONDUTOR,3701,247.0
1,REAÇÃO TARDIA OU INEFICIENTE DO CONDUTOR,3476,181.0
2,ACESSAR A VIA SEM OBSERVAR A PRESENÇA DOS OUTR...,2270,133.0
3,CONDUTOR DEIXOU DE MANTER DISTÂNCIA DO VEÍCULO...,1387,38.0
4,VELOCIDADE INCOMPATÍVEL,1310,155.0


trilha B

In [33]:
inicio = time.time()

dl_causa = gold_causa.copy()

tempo_dl = time.time() - inicio

dl_causa.head()

,causa_acidente,acidentes,mortos,feridos
0,AUSÊNCIA DE REAÇÃO DO CONDUTOR,3701,247,4170
1,REAÇÃO TARDIA OU INEFICIENTE DO CONDUTOR,3476,181,4021
2,ACESSAR A VIA SEM OBSERVAR A PRESENÇA DOS OUTR...,2270,133,2915
3,CONDUTOR DEIXOU DE MANTER DISTÂNCIA DO VEÍCULO...,1387,38,1594
4,VELOCIDADE INCOMPATÍVEL,1310,155,1712


# Comparação

In [34]:
print("Tempo DW:", tempo_dw)
print("Tempo DL:", tempo_dl)

Tempo DW: 0.008468866348266602
Tempo DL: 0.0002548694610595703


# Pergunta 4

Quais municípios concentram mais acidentes?

trilha A

In [35]:
inicio = time.time()

consulta = """

SELECT

    l.municipio,

    COUNT(*) AS acidentes,

    SUM(f.mortos) AS mortos

FROM fato_acidentes f

JOIN dim_local l

ON f.sk_local = l.sk_local

GROUP BY l.municipio

ORDER BY acidentes DESC

LIMIT 10

"""

dw_municipio = con.execute(consulta).fetchdf()

tempo_dw = time.time() - inicio


dw_municipio.head()

,municipio,acidentes,mortos
0,BRASILIA,327,18.0
1,DUQUE DE CAXIAS,266,12.0
2,SAO JOSE,256,3.0
3,RECIFE,245,16.0
4,BETIM,232,6.0


trilha B

In [36]:
inicio = time.time()

dl_municipio = gold_municipio.copy()

tempo_dl = time.time() - inicio

dl_municipio.head()

,municipio,acidentes,mortos
0,BRASILIA,327,18
1,DUQUE DE CAXIAS,266,12
2,SAO JOSE,256,3
3,RECIFE,245,16
4,BETIM,232,6


# Comparação

In [37]:
print("Tempo DW:", tempo_dw)
print("Tempo DL:", tempo_dl)

Tempo DW: 0.010675430297851562
Tempo DL: 0.00042724609375


# Pergunta 5
Em quais dias da semana ocorrem mais acidentes?

trilha A

In [38]:
inicio = time.time()

consulta = """

SELECT

    d.dia_semana,

    COUNT(*) AS acidentes,

    SUM(f.mortos) AS mortos,

    SUM(f.feridos) AS feridos

FROM fato_acidentes f

JOIN dim_data d

ON f.sk_data = d.sk_data

GROUP BY d.dia_semana

ORDER BY acidentes DESC

"""

dw_dia = con.execute(consulta).fetchdf()

tempo_dw = time.time() - inicio

dw_dia.head()

,dia_semana,acidentes,mortos,feridos
0,DOMINGO,3657,342.0,4327.0
1,SÁBADO,3605,318.0,4261.0
2,SEXTA-FEIRA,3578,280.0,4218.0
3,QUINTA-FEIRA,3396,247.0,3938.0
4,SEGUNDA-FEIRA,3168,265.0,3710.0


trilha B

In [39]:
inicio = time.time()

dl_dia = gold_dia.copy()

tempo_dl = time.time() - inicio

print("Tempo DL:", tempo_dl)

dl_dia.head()

Tempo DL: 0.0002930164337158203


,dia_semana,acidentes,mortos,feridos
0,DOMINGO,3657,342,4327
1,QUARTA-FEIRA,3093,229,3529
2,QUINTA-FEIRA,3396,247,3938
3,SEGUNDA-FEIRA,3168,265,3710
4,SEXTA-FEIRA,3578,280,4218


# Comparação

In [40]:
print("Tempo DW:", tempo_dw)
print("Tempo DL:", tempo_dl)

Tempo DW: 0.0079803466796875
Tempo DL: 0.0002930164337158203
